# 83 — Historical Site-Year Fusion (fallback-aware replacement)

This replacement can rebuild the emissions warehouse directly from `emissions_long.parquet` if notebook 81 did not create a usable CSV.

In [ ]:
import os, io, re, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def safe_read_parquet(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        import pyarrow.parquet as pq
        df = pq.read_table(p).to_pandas()
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def clean_facility(fn: str) -> str:
    s = re.sub(r"\.xlsm$|\.xlsx$|\.pdf$", "", str(fn), flags=re.I)
    s = re.sub(r"^[A-Z0-9]+\s+", "", s)
    s = re.sub(r"\s+Annual Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Annual Performance Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Report 2024$", "", s, flags=re.I)
    s = re.sub(r"\s+2024$", "", s)
    return s.strip()

In [ ]:
step = "83_historical_site_year_fusion"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

wh, wh_meta = safe_read_csv(OUTPUTS / "81_historical_emissions_warehouse" / "historical_emissions_warehouse.csv")
nc, nc_meta = safe_read_csv(OUTPUTS / "82_documentary_noncompliance_warehouse" / "documentary_noncompliance_warehouse.csv")

if wh.empty:
    emissions, em_meta = safe_read_parquet(DATA / "emissions_long.parquet")
    registry, _ = safe_read_csv(OUTPUTS / "80_master_permit_registry" / "master_permit_registry.csv")
    if emissions.empty:
        raise FileNotFoundError("Need outputs/81_historical_emissions_warehouse/historical_emissions_warehouse.csv or data_sources/emissions_long.parquet")

    work = emissions.copy()
    work["permit_id"] = work.get("permit_id", pd.Series(dtype=str)).astype(str).str.strip()
    work["report_year"] = pd.to_numeric(work.get("report_year", work.get("year", np.nan)), errors="coerce")
    work["pollutant_std"] = work.get("pollutant", work.get("parameter", work.get("analyte", ""))).astype(str).str.strip()
    work["metric_std"] = work.get("metric", "").astype(str).str.strip()
    work["value_num"] = pd.to_numeric(work.get("value", work.get("result", np.nan)), errors="coerce")
    work["unit_std"] = work.get("unit", work.get("units", "")).astype(str).str.strip()
    if not registry.empty:
        work = work.merge(registry[["permit_id","facility_name","site_id","region_folder"]].drop_duplicates(), on="permit_id", how="left")
    else:
        work["facility_name"] = work["permit_id"]
        work["site_id"] = np.nan
        work["region_folder"] = ""
    work = work[
        work["permit_id"].astype(str).str.len().gt(0)
        & work["report_year"].notna()
        & work["pollutant_std"].astype(str).str.len().gt(0)
        & work["metric_std"].astype(str).str.len().gt(0)
    ].copy()
    wh = (
        work.groupby(
            ["permit_id","facility_name","site_id","region_folder","report_year","pollutant_std","metric_std","unit_std"],
            as_index=False
        )
        .agg(value_mean=("value_num","mean"), value_max=("value_num","max"), row_count=("permit_id","size"))
    )
    wh_meta = {"path": str(DATA / "emissions_long.parquet"), "status": "rebuilt_from_raw_emissions"}

if wh.empty:
    raise RuntimeError("Historical site-year fusion has no emissions warehouse rows to work from")

group_keys = ["permit_id","facility_name","site_id","region_folder","report_year"]

exceedance_like = wh[wh["metric_std"].astype(str).str.contains("exceedance_count", case=False, na=False)].copy()
elv = wh[wh["metric_std"].astype(str) == "ELV"].copy()
means = wh[wh["metric_std"].astype(str).isin(["daily_mean","half_hourly_mean","p95","max"])].copy()

if exceedance_like.empty:
    ex_perm = wh[group_keys].drop_duplicates().copy()
    ex_perm["exceedance_total"] = 0.0
else:
    ex_perm = exceedance_like.groupby(group_keys, as_index=False)["value_max"].sum().rename(columns={"value_max":"exceedance_total"})

if means.empty or elv.empty:
    ratio_sum = wh[group_keys].drop_duplicates().copy()
    ratio_sum["worst_ratio"] = 0.0
else:
    ratio = means.merge(
        elv[["permit_id","pollutant_std","report_year","value_mean"]].rename(columns={"value_mean":"elv_value"}),
        on=["permit_id","pollutant_std","report_year"], how="left"
    )
    ratio["ratio"] = ratio["value_max"] / ratio["elv_value"]
    ratio_sum = ratio.groupby(group_keys, as_index=False)["ratio"].max().rename(columns={"ratio":"worst_ratio"})

base = wh.groupby(group_keys, as_index=False).agg(
    pollutant_count=("pollutant_std","nunique"),
    metric_rows=("metric_std","size")
)
base = base.merge(ex_perm, on=group_keys, how="left")
base = base.merge(ratio_sum, on=group_keys, how="left")

if not nc.empty:
    nc2 = nc[["permit_id","report_year","cems_hit_count","noncompliance_hits","incident_hits"]].copy()
    base = base.merge(nc2, on=["permit_id","report_year"], how="left")

for c in ["exceedance_total","worst_ratio","cems_hit_count","noncompliance_hits","incident_hits"]:
    if c not in base.columns:
        base[c] = 0
    base[c] = pd.to_numeric(base[c], errors="coerce").fillna(0)

base["historical_risk_score"] = (
    base["exceedance_total"] * 3
    + base["worst_ratio"] * 10
    + base["noncompliance_hits"] * 0.3
    + base["incident_hits"] * 0.2
    + base["cems_hit_count"] * 0.01
)

base.to_csv(out / "historical_site_year_fusion.csv", index=False)
write_json(out / "historical_site_year_fusion_summary.json", {
    "rows": int(len(base)),
    "permits": int(base["permit_id"].nunique()),
    "years": int(base["report_year"].nunique()),
})

manifest = manifest_base(step, [CONFIGS / "permit_registry.yml"])
manifest["inputs"] = {"warehouse_source": wh_meta, "noncompliance_source": nc_meta}
add_artifact(manifest, out / "historical_site_year_fusion.csv")
add_artifact(manifest, out / "historical_site_year_fusion_summary.json")
write_json(out / "manifest.json", manifest)
display(base.head(20))